# Avance 1 – Análisis Exploratorio de Datos (EDA) | ML–SEDP
**Corredor CDMX–Acapulco (MEX-095D) · Unidad de análisis: CTN (Candidate Territorial Node)**

Este notebook explora `03_PROCESSED/CTN_Master.csv`, generado por el pipeline reproducible `src/s01…s07`.
Cada fila es un CTN sobre la geometría real de la RNC (chainage de red cada ~10 km + puntos estratégicos).

**Reglas que respetamos en todo el análisis (guía ML–SEDP §9):**
1. No existe etiqueta de "sitio ideal": el EDA no construye ni sugiere una variable objetivo de idoneidad.
2. Variables `mun_*` describen el **municipio**, no el predio del CTN.
3. Cada fuente conserva su periodo (`feature_provenance.csv`); no se mezclan años sin registrarlo.
4. Los CTN vecinos se parecen (autocorrelación espacial): lo medimos aquí para justificar validación espacial.
5. Una tendencia en el EDA no es una decisión de ingeniería.

**Estructura:** 1. Estructura de los datos · 2. Análisis univariante · 3. Tendencias temporales · 4. Análisis bi/multivariante · 5. Preprocesamiento · 6. Selección preliminar de características · 7. Conclusiones

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100, "display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "04_NOTEBOOKS" else Path.cwd()
P, I, D = ROOT / "03_PROCESSED", ROOT / "02_INTERIM", ROOT / "00_DOCUMENTATION"
FIG = ROOT / "06_RESULTS" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

df = pd.read_csv(P / "CTN_Master.csv", dtype={"cve_mun": str})
prov = pd.read_csv(D / "feature_provenance.csv")
print(f"CTN_Master: {df.shape[0]} CTN x {df.shape[1]} columnas")
df.head()

## 1. Estructura de los datos
### 1.1 Forma, tipos y rol de cada columna
Separamos **identificadores/metadatos** (no se analizan como variables), **numéricas** y **categóricas**.

In [ ]:
ID_META = ["ctn_id", "chainage_m", "geometry_status", "snap_distance_m", "lon", "lat", "top_id",
           "top_locating_method", "mun_assignment_method", "cve_mun", "strategic_label", "tdpa_year",
           "spatial_block"]
meta = [c for c in ID_META if c in df.columns]
cat_cols = [c for c in df.columns if c not in meta and not pd.api.types.is_numeric_dtype(df[c])]
num_cols = [c for c in df.columns if c not in meta + cat_cols]
print(f"Metadatos: {len(meta)} | Numéricas: {len(num_cols)} | Categóricas: {len(cat_cols)}")

tipos = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(),
                      "rol": ["metadato" if c in meta else "categórica" if c in cat_cols else "numérica"
                              for c in df.columns]})
tipos = tipos.join(prov.set_index("feature_name")[["class", "source", "source_period", "spatial_resolution"]])
tipos

### 1.2 Estadísticas descriptivas (todas las variables numéricas)

In [ ]:
desc = df[num_cols].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc["cv"] = desc["std"] / desc["mean"].abs()
desc["skew"] = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()
desc.round(3)

### 1.3 Variables categóricas: frecuencia de clases y cardinalidad

In [ ]:
for c in cat_cols:
    vc = df[c].value_counts(dropna=False)
    print(f"\n{c}  (cardinalidad = {df[c].nunique()})")
    print(pd.DataFrame({"n": vc, "%": (vc / len(df) * 100).round(1)}).to_string())

Cardinalidad en las **fuentes crudas** (antes de agregar a CTN). Aquí aparece la *alta cardinalidad* real:
clases de actividad DENUE y cultivos SIAP.

In [ ]:
src_tables = {}
for name, f in {"denue": "denue_establecimientos.csv", "dgsiap": "dgsiap_cultivos_municipios_corredor.csv",
                "censo": "censo_localidades_corredor.csv"}.items():
    if (I / f).exists():
        src_tables[name] = pd.read_csv(I / f, low_memory=False)
card = []
if "denue" in src_tables:
    t = src_tables["denue"]
    card += [("DENUE", "Clase_actividad", t["Clase_actividad"].nunique(), len(t)),
             ("DENUE", "scian (6 díg.)", t["scian"].nunique(), len(t)),
             ("DENUE", "sector_2d", t["sector_2d"].nunique(), len(t))]
if "dgsiap" in src_tables:
    t = src_tables["dgsiap"]
    card += [("SIAP", "cultivo", t["cultivo"].nunique(), len(t)), ("SIAP", "modalidad", t["modalidad"].nunique(), len(t))]
if "censo" in src_tables:
    t = src_tables["censo"]
    card += [("Censo", "NOM_LOC", t["NOM_LOC"].nunique(), len(t))]
pd.DataFrame(card, columns=["fuente", "variable", "cardinalidad", "registros"])

### 1.4 Valores faltantes y patrones de ausencia
Distinguimos tres tipos: **cero real** (p. ej. 0 establecimientos en 5 km), **faltante estructural**
(p. ej. TDPA cuando no hay estación de aforo a ≤25 km en red) y **faltante por confidencialidad** (`*` en el Censo).

In [ ]:
miss = pd.DataFrame({"n_faltantes": df[num_cols + cat_cols].isna().sum(),
                     "%": (df[num_cols + cat_cols].isna().mean() * 100).round(1)})
miss = miss[miss["n_faltantes"] > 0].sort_values("%", ascending=False)
display(miss.join(prov.set_index("feature_name")[["source", "spatial_resolution"]]))

fig, ax = plt.subplots(figsize=(12, 5))
cols_m = miss.index.tolist()
if cols_m:
    sns.heatmap(df.set_index("chainage_km")[cols_m].isna().T, cbar=False, cmap="Greys", ax=ax)
    ax.set_title("Patrón de faltantes a lo largo del corredor (negro = faltante)")
    ax.set_xlabel("chainage (km desde CDMX)")
savefig("01_missing_pattern_chainage")

In [ ]:
# ¿La ausencia es aleatoria? Comparamos variables observadas en CTN con y sin dato (prueba Mann-Whitney)
tests = []
for mcol in [c for c in ["tdpa", "mun_agri_sown_ha"] if c in df.columns and df[c].isna().any()]:
    flag = df[mcol].isna()
    for v in [c for c in ["chainage_km", "pop_10km", "denue_n_est_5km", "rural_pop_share_10km"] if c in df.columns]:
        a, b = df.loc[flag, v].dropna(), df.loc[~flag, v].dropna()
        if len(a) > 2 and len(b) > 2:
            tests.append([mcol, v, a.median(), b.median(), stats.mannwhitneyu(a, b).pvalue])
pd.DataFrame(tests, columns=["faltante_en", "comparada", "mediana_si_falta", "mediana_no_falta", "p_value"]).round(4)

*Interpretación (completar con los resultados reales):* si `p < 0.05`, la ausencia **no es aleatoria**
(MAR/MNAR); por ejemplo, el TDPA falta en tramos sin estaciones de aforo y los datos agrícolas faltan en
municipios urbanos. Esto define la estrategia de imputación de la sección 5.

## 2. Análisis univariante
### 2.1 Variables numéricas: histogramas y boxplots

In [ ]:
key_num = [c for c in num_cols if not c.endswith(("_min_month", "_missing_pct", "_p10", "_p90"))]
n = len(key_num); ncols = 4; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.2 * nrows))
for ax, c in zip(axes.flat, key_num):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color="#2a7f8f")
    ax.set_title(f"{c}\nskew={df[c].skew():.2f}", fontsize=9); ax.set_xlabel("")
for ax in axes.flat[n:]: ax.axis("off")
savefig("02_hist_numeric")

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6 * nrows))
for ax, c in zip(axes.flat, key_num):
    sns.boxplot(x=df[c], ax=ax, color="#c9a227"); ax.set_title(c, fontsize=9); ax.set_xlabel("")
for ax in axes.flat[n:]: ax.axis("off")
savefig("03_box_numeric")

### 2.2 Variables categóricas: gráficos de barras

In [ ]:
plot_cats = [c for c in cat_cols if df[c].nunique() <= 30]
if plot_cats:
    fig, axes = plt.subplots(1, len(plot_cats), figsize=(6 * len(plot_cats), 4), squeeze=False)
    for ax, c in zip(axes.flat, plot_cats):
        df[c].fillna("FALTANTE").value_counts().plot.barh(ax=ax, color="#4f7f3a"); ax.set_title(c)
    savefig("04_bar_categorical")
if "denue" in src_tables:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    src_tables["denue"]["Clase_actividad"].value_counts().head(20).plot.barh(ax=axes[0], color="#1f4e79")
    axes[0].set_title("DENUE: 20 clases de actividad más frecuentes (alta cardinalidad)")
    src_tables["denue"]["sector_2d"].astype(str).value_counts().plot.bar(ax=axes[1], color="#2a7f8f")
    axes[1].set_title("DENUE agrupado a sector SCIAN (2 dígitos)")
    savefig("05_bar_denue_cardinality")
if "dgsiap" in src_tables:
    t = src_tables["dgsiap"].groupby("cultivo")["sembrada_ha"].sum().sort_values(ascending=False)
    t.head(20).plot.barh(figsize=(8, 6), color="#4f7f3a", title="SIAP: superficie sembrada por cultivo (municipios del corredor)")
    savefig("06_bar_crops")

### 2.3 Sesgo y transformación no lineal
Variables de conteo y población suelen tener sesgo positivo fuerte (CDMX vs. localidades rurales de Guerrero).
Comparamos el sesgo antes y después de `log1p`.

In [ ]:
sk = pd.DataFrame({"skew_original": df[key_num].skew()})
pos = [c for c in key_num if (df[c].dropna() >= 0).all()]
sk.loc[pos, "skew_log1p"] = np.log1p(df[pos]).skew()
sk["transformar"] = (sk["skew_original"].abs() > 1) & (sk["skew_log1p"].abs() < sk["skew_original"].abs())
SKEWED = sk.index[sk["transformar"]].tolist()
display(sk.round(2).sort_values("skew_original", ascending=False))
print("Candidatas a log1p:", SKEWED)
if "censo" in src_tables:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(src_tables["censo"]["POBTOT"], ax=axes[0]); axes[0].set_title("POBTOT por localidad (original)")
    sns.histplot(np.log1p(src_tables["censo"]["POBTOT"]), ax=axes[1]); axes[1].set_title("log1p(POBTOT)")
    savefig("07_censo_log_transform")

## 3. Tendencias temporales
El CTN_Master es una foto transversal, pero NASA POWER trae una serie mensual 2015–2024 y el TDPA tiene año de observación.

In [ ]:
f = I / "nasa_power_monthly.csv"
if f.exists():
    nm = pd.read_csv(f)
    clim = nm.groupby(["parameter", "month"])["value"].agg(["mean", "std"]).reset_index()
    params = clim["parameter"].unique()
    fig, axes = plt.subplots(1, len(params), figsize=(4 * len(params), 3.5), squeeze=False)
    for ax, p in zip(axes.flat, params):
        c = clim[clim.parameter == p]
        ax.plot(c.month, c["mean"], marker="o"); ax.fill_between(c.month, c["mean"] - c["std"], c["mean"] + c["std"], alpha=.2)
        ax.set_title(p, fontsize=9); ax.set_xticks(range(1, 13))
    savefig("08_nasa_climatology")
    ann = nm[nm.parameter == "ALLSKY_SFC_SW_DWN"].groupby(["ctn_id", "year"])["value"].mean().reset_index()
    sns.lineplot(data=ann, x="year", y="value", estimator="mean", errorbar="sd")
    plt.title("Radiación anual media de los CTN (kWh/m²/día) ± 1 sd"); savefig("09_nasa_annual_trend")
    slope = stats.linregress(ann.groupby("year")["value"].mean().index, ann.groupby("year")["value"].mean().values)
    print(f"Tendencia lineal de radiación: {slope.slope:.4f} kWh/m²/día por año (p={slope.pvalue:.3f})")
if "tdpa_year" in df:
    print("Años de TDPA asignados:", df["tdpa_year"].value_counts(dropna=False).to_dict())

## 4. Análisis bi/multivariante
### 4.1 Perfil longitudinal del corredor
Cómo cambian las variables clave con la distancia desde CDMX: la forma más natural de leer un corredor.

In [ ]:
prof = [c for c in ["pop_10km", "denue_n_est_5km", "tdpa", "allsky_sfc_sw_dwn_mean", "t2m_mean",
                    "rural_pop_share_10km", "mun_agri_sown_ha"] if c in df.columns]
fig, axes = plt.subplots(len(prof), 1, figsize=(13, 2.1 * len(prof)), sharex=True)
for ax, c in zip(np.atleast_1d(axes), prof):
    ax.plot(df["chainage_km"], df[c], marker="o", ms=3, color="#1f4e79"); ax.set_ylabel(c, fontsize=8)
np.atleast_1d(axes)[-1].set_xlabel("chainage (km desde CDMX)")
savefig("10_corridor_profile")

### 4.2 Correlación cuantitativa (Spearman, robusta a sesgo y atípicos) y Pearson sobre log1p

In [ ]:
corr_cols = [c for c in key_num if df[c].nunique() > 2]
sp = df[corr_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(sp, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"shrink": .6})
ax.set_title("Correlación de Spearman entre features de CTN"); savefig("11_corr_spearman")

X_log = df[corr_cols].copy()
X_log[[c for c in SKEWED if c in X_log]] = np.log1p(X_log[[c for c in SKEWED if c in X_log]])
pe = X_log.corr(method="pearson")
pairs = (sp.where(np.triu(np.ones(sp.shape), 1).astype(bool)).stack().rename("spearman").to_frame()
         .join(pe.stack().rename("pearson_log")))
pairs["abs"] = pairs["spearman"].abs()
display(pairs.sort_values("abs", ascending=False).head(20).round(3))

In [ ]:
# Significancia con n pequeño: con ~30-40 CTN sólo |rho| > ~0.35 es significativo al 5%.
n_ctn = len(df)
r_crit = np.tanh(1.96 / np.sqrt(n_ctn - 3))
print(f"n = {n_ctn} CTN -> |rho| crítico aprox. (alpha=0.05): {r_crit:.2f}")

### 4.3 Relaciones gráficas entre variables importantes

In [ ]:
pp = [c for c in ["pop_10km", "denue_n_est_5km", "tdpa", "allsky_sfc_sw_dwn_mean", "rural_pop_share_10km"] if c in df.columns]
g = sns.pairplot(np.log1p(df[pp].clip(lower=0)).assign(bloque=df["spatial_block"].astype(str)),
                 hue="bloque", corner=True, plot_kws={"s": 25})
g.figure.suptitle("Pairplot (log1p) coloreado por bloque espacial de 50 km", y=1.02)
plt.savefig(FIG / "12_pairplot.png", dpi=150, bbox_inches="tight"); plt.show()

### 4.4 Distribución por categorías (análisis bivariado)

In [ ]:
by = [c for c in ["ctn_type", "mun_agri_top_crop", "spatial_block"] if c in df.columns]
targets = [c for c in ["pop_10km", "denue_n_est_5km", "allsky_sfc_sw_dwn_mean"] if c in df.columns]
fig, axes = plt.subplots(len(by), len(targets), figsize=(5 * len(targets), 3.8 * len(by)), squeeze=False)
for i, b in enumerate(by):
    for j, t in enumerate(targets):
        sns.boxplot(data=df, x=b, y=t, ax=axes[i, j], color="#9bc4cb")
        sns.stripplot(data=df, x=b, y=t, ax=axes[i, j], color="k", size=3)
        axes[i, j].tick_params(axis="x", rotation=45); axes[i, j].set_title(f"{t} por {b}", fontsize=9)
savefig("13_box_by_category")
for b in by:
    for t in targets:
        groups = [g[t].dropna() for _, g in df.groupby(b) if g[t].notna().sum() > 1]
        if len(groups) > 1:
            print(f"Kruskal-Wallis {t} ~ {b}: p = {stats.kruskal(*groups).pvalue:.4f}")

### 4.5 Autocorrelación espacial a lo largo del corredor
Si un CTN se parece mucho a su vecino, una división aleatoria train/test inflaría las métricas
(error #4 de la guía). Medimos la correlación entre cada CTN y el siguiente (lag 1) y a distancias mayores.

In [ ]:
ordered = df.sort_values("chainage_km")
ac = {}
for c in [c for c in corr_cols if ordered[c].notna().sum() > 10]:
    s = ordered[c]
    ac[c] = {f"lag{k}": s.corr(s.shift(-k), method="spearman") for k in (1, 3, 5)}
ac = pd.DataFrame(ac).T.sort_values("lag1", ascending=False)
display(ac.round(2))
ac[["lag1", "lag3", "lag5"]].head(15).plot.barh(figsize=(8, 6), title="Autocorrelación espacial (Spearman) por rezago de CTN")
savefig("14_spatial_autocorrelation")

## 5. Preprocesamiento (ejecutado y justificado)
> El preprocesamiento aquí es **exploratorio**. En la fase de modelado, imputación, escalado y transformaciones
> se ajustarán **sólo con datos de entrenamiento dentro de cada fold espacial** para evitar leakage.

### 5.1 Valores faltantes
| Caso | Decisión | Justificación |
|---|---|---|
| Conteos (`denue_n_*`, `pop_*km`, `n_localities_*`) en 0 | Se conservan como 0 | Es un valor real si la consulta fue exitosa |
| `tdpa`, `tdpa_pct_*` sin TOP a ≤25 km en red | Interpolación lineal **a lo largo del chainage** entre TOP + indicador `tdpa_imputed` | El tránsito en un corredor cambia de forma continua entre entronques; la interpolación por red respeta TOP ≠ CTN. Clase P (proxy) |
| `mun_agri_*` en municipios sin registro SIAP | Indicador `mun_agri_missing` + imputación con el CTN vecino más cercano en chainage | Faltante estructural (municipios urbanos); se marca para no confundirlo con observación |
| `schooling_mean_10km` (`*` confidencial) | Mediana del corredor + indicador | Pocos casos; confidencialidad, no ausencia territorial |

In [ ]:
pre = df.copy()
pre_log = []
def impute_chainage(frame, col):
    s = frame.sort_values("chainage_km").set_index("chainage_km")[col]
    filled = s.interpolate(method="index", limit_direction="both")
    return frame.sort_values("chainage_km").assign(**{col: filled.values}).sort_index()

for col in [c for c in ["tdpa", "tdpa_pct_heavy", "tdpa_pct_bus"] if c in pre and pre[c].isna().any()]:
    pre["tdpa_imputed"] = pre["tdpa"].isna().astype(int)
    n0 = pre[col].isna().sum(); pre = impute_chainage(pre, col)
    pre_log.append([col, n0, "interpolación lineal por chainage"])

agri = [c for c in pre.columns if c.startswith("mun_agri_")]
if agri and pre[agri[0]].isna().any():
    pre["mun_agri_missing"] = pre[agri[0]].isna().astype(int)
    o = pre.sort_values("chainage_km")
    for c in agri:
        n0 = o[c].isna().sum()
        o[c] = o[c].ffill().bfill() if not pd.api.types.is_numeric_dtype(o[c]) else o[c].interpolate(method="nearest").ffill().bfill()
        pre_log.append([c, n0, "vecino más cercano en chainage + indicador"])
    pre = o.sort_index()

for c in [c for c in num_cols if c in pre and pre[c].isna().any()]:
    n0 = pre[c].isna().sum(); pre[f"{c}_missing"] = pre[c].isna().astype(int)
    pre[c] = pre[c].fillna(pre[c].median()); pre_log.append([c, n0, "mediana + indicador"])
display(pd.DataFrame(pre_log, columns=["variable", "n_imputados", "estrategia"]))
print("Faltantes restantes en numéricas:", int(pre[num_cols].isna().sum().sum()))

### 5.2 Valores atípicos
Detectamos con IQR (1.5×) y z-robusto (MAD). **No eliminamos CTN atípicos**: en un corredor CDMX–Acapulco
los extremos (zona metropolitana, Chilpancingo, Acapulco) son heterogeneidad territorial real, no error.
Estrategia: `log1p` para variables sesgadas y escalado robusto (mediana/IQR) en el modelado; winsorización p1–p99 sólo si un modelo sensible lo requiere.

In [ ]:
out_rows = []
for c in key_num:
    s = pre[c].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    n_iqr = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    mad = stats.median_abs_deviation(s, scale="normal")
    n_mad = (np.abs((s - s.median()) / mad) > 3.5).sum() if mad > 0 else 0
    if n_iqr or n_mad:
        ids = pre.loc[s.index[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)], "ctn_id"].tolist()
        out_rows.append([c, n_iqr, n_mad, ids[:6]])
outl = pd.DataFrame(out_rows, columns=["variable", "atípicos_IQR", "atípicos_MAD", "CTN (muestra)"])
display(outl)
for c in SKEWED:
    pre[f"log1p_{c}"] = np.log1p(pre[c].clip(lower=0))
print("Columnas log1p creadas:", [f"log1p_{c}" for c in SKEWED])

### 5.3 Alta cardinalidad
- **DENUE `Clase_actividad`** (cientos de clases) → agrupada a **sector SCIAN de 2 dígitos** y a conteos temáticos
  relevantes para un Green Stop (alimentos/hospedaje, comercio, gasolineras, reparación automotriz) + índice de diversidad de Shannon.
- **Cultivos SIAP** → resumidos en número de cultivos, diversidad de Shannon y cultivo principal; en `mun_agri_top_crop`
  se agrupan los cultivos poco frecuentes en `OTROS` (umbral: < 2 CTN).
- **NOM_LOC** (nombres de localidad) → no se usa como feature; sólo como etiqueta.

In [ ]:
if "mun_agri_top_crop" in pre:
    vc = pre["mun_agri_top_crop"].value_counts()
    rare = vc[vc < 2].index
    pre["mun_agri_top_crop_grp"] = pre["mun_agri_top_crop"].where(~pre["mun_agri_top_crop"].isin(rare), "OTROS")
    print("Cardinalidad cultivo principal:", pre["mun_agri_top_crop"].nunique(), "->", pre["mun_agri_top_crop_grp"].nunique())

## 6. Selección preliminar de características (objetivo 2.1)
Criterios no supervisados (no hay variable objetivo, por diseño):
1. **Varianza casi nula** (p. ej. variables NASA idénticas entre CTN que caen en la misma celda de malla).
2. **Redundancia**: de cada par con |Spearman| > 0.90 se conserva la de mayor completitud original / interpretabilidad.
3. **Metadatos y riesgos de leakage** excluidos (`tdpa_year`, `ctn_type`, indicadores de método).

In [ ]:
cand = [c for c in pre.columns if c in num_cols and c not in ("chainage_km",)]
cand = [c for c in cand if not c.endswith(("_missing_pct", "_min_month"))]
var_rel = (pre[cand].std() / pre[cand].mean().abs().replace(0, np.nan))
low_var = var_rel[var_rel < 0.01].index.tolist()
keep = [c for c in cand if c not in low_var]
compl = df[keep].notna().mean()
sp2 = pre[keep].corr(method="spearman").abs()
drop_red = set()
for i, a in enumerate(keep):
    for b in keep[i + 1:]:
        if a in drop_red or b in drop_red: continue
        if sp2.loc[a, b] > 0.90:
            drop_red.add(b if compl[a] >= compl[b] else a)
selected = [c for c in keep if c not in drop_red]
sel = pd.DataFrame({"feature": cand,
                    "decisión": ["descartada: varianza ~0" if c in low_var else "descartada: redundante (|rho|>0.9)"
                                 if c in drop_red else "seleccionada" for c in cand]})
sel = sel.merge(prov[["feature_name", "class", "source", "spatial_resolution"]], left_on="feature",
                right_on="feature_name", how="left").drop(columns="feature_name")
display(sel.sort_values("decisión"))
print(f"{len(cand)} candidatas -> {len(selected)} seleccionadas")

In [ ]:
out_cols = ["ctn_id", "chainage_km", "lon", "lat", "spatial_block", "cve_mun"] + selected + \
           [c for c in pre.columns if c.endswith(("_imputed", "_missing")) or c.startswith("log1p_") or c.endswith("_grp")]
out_cols = list(dict.fromkeys(c for c in out_cols if c in pre.columns))
pre[out_cols].to_csv(P / "CTN_Master_preprocessed.csv", index=False)
sel.to_csv(ROOT / "06_RESULTS" / "feature_selection_eda.csv", index=False)
print("Guardado: 03_PROCESSED/CTN_Master_preprocessed.csv", pre[out_cols].shape)

## 7. Conclusiones del EDA
> Completar con los números reales una vez corrido el pipeline. Se dejan las preguntas guía de la actividad
> y lo que el código ya calcula para responderlas.

**¿Hay valores faltantes? ¿Patrones de ausencia?** Ver §1.4. Esperado: TDPA estructuralmente faltante donde no
hay estación de aforo; datos agrícolas ausentes en alcaldías/municipios urbanos. Las pruebas de Mann-Whitney indican
si la ausencia depende de urbanización (no aleatoria).

**Estadísticas resumidas.** §1.2: rango de población a 10 km, establecimientos a 5 km, radiación media (kWh/m²/día).

**¿Atípicos?** §5.2: los extremos corresponden a la zona metropolitana y a ciudades intermedias; son reales y se conservan.

**Cardinalidad.** §1.3: DENUE y SIAP tienen alta cardinalidad en origen; se resolvió por agregación sectorial e índices de diversidad.

**¿Sesgo? ¿Transformación?** §2.3: conteos y población requieren `log1p`; variables climáticas son aproximadamente simétricas.

**¿Tendencias temporales?** §3: estacionalidad marcada de radiación (mínimo en temporada de lluvias) y tendencia anual (pendiente y p-value reportados).

**¿Correlaciones?** §4.2–4.3: no hay variable dependiente por diseño (no se entrena "sitio ideal"); se reportan relaciones
entre independientes. Esperado: bloque urbano (población–establecimientos–TDPA) correlacionado positivamente y opuesto a ruralidad;
radiación y temperatura aumentando hacia el sur. Con n≈30–40, sólo |rho| > umbral crítico es interpretable.

**Distribución por categorías.** §4.4 (Kruskal-Wallis por bloque espacial y tipo de CTN).

**Autocorrelación espacial.** §4.5: si lag1 es alto, confirma que la validación deberá ser por bloques del corredor.

**¿Normalizar imágenes? / ¿Desequilibrio de clases?** No aplican en este avance: no hay imágenes (NDVI de Copernicus
se integrará después como estadísticos por AOI) y no hay variable objetivo de clase.

**Limitaciones.** Variables `mun_*` son contexto municipal, no predial; NASA POWER es malla regional; DENUE subregistra
informalidad; TDPA proviene de años distintos (se conserva `tdpa_year`); no hay datos públicos de capacidad de red eléctrica.